# 06 · Source Fitting

**Audience:** Lila / Professor &nbsp;|&nbsp; **Time:** ~30 min &nbsp;|&nbsp; **Prerequisites:** [01 · Quickstart](01-quickstart.ipynb)

## What you'll learn

- What source fitting is and when to use it
- How to define a `MagneticSource` (ROI, pixel spacing, dipole model priors)
- Quick path (Lila): define one source, call `fit_sources()`, read the moment
- Deep path (Professor): inspect convergence, compare predicted vs measured Bz, visualise residuals
- Multi-source fitting for non-overlapping regions of interest

## This notebook uses synthetic data

All examples use `qdmpy.make_synthetic_qdm_result()` so no MATLAB files or GPU
hardware are required. The synthetic result contains a dipole-like B111 field
pattern, which is used as the Bz input to the source fitter.

---

In [ ]:
%matplotlib inline
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

import qdmpy

## 1. What is source fitting?

Source fitting extracts the physical parameters of a magnetic source —
moment magnitude, inclination, and declination — by fitting a theoretical
dipole field to the measured Bz map over a defined region of interest (ROI).

**Key concepts:**

| Concept | Description |
|---------|-------------|
| **`MagneticSource`** | Defines the ROI (center pixel, half-extent) and initial dipole model priors |
| **`MagneticModel`** | Three parameters: inclination (deg), declination (deg), magnetic_moment (A·m²) |
| **standoff_m** | Sensor-to-sample distance in metres; initial guess for the z-position of the dipole |
| **`fit_sources()`** | Runs scipy TRF least-squares (Huber loss) on each `MagneticSource` in `QDMResult.field_sources` |
| **`FitSourceResult`** | Contains updated `MagneticSource` (fitted params) + raw `scipy.OptimizeResult` |

**Declination convention** (pypole):
dec=0 points toward -Y (image south), increasing counterclockwise:
dec=90 → +X (East), dec=180 → +Y (North), dec=270 → -X (West).

---

In [ ]:
# Create a synthetic QDMResult with a dipole-like B111 pattern
result = qdmpy.make_synthetic_qdm_result(shape=(64, 64), pixel_spacing=4e-6)
print(result)
print(f"B111 shape: {result.b111_remanent.shape}")
print(f"Pixel spacing: {result.pixel_spacing * 1e6:.0f} µm")

In [ ]:
# Visualise B111 and the Bz component from the 3D reconstruction
b111 = result.b111_remanent          # (64, 64) in µT
mm = result.magnetic_map             # triggers Fourier reconstruction
bz_uT = mm.bz.values                # (64, 64) in µT

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, data, title in zip(
    axes,
    [b111, bz_uT],
    ['B111 remanent (µT)', 'Bz reconstructed (µT)'],
    strict=False,
):
    vmax = np.percentile(np.abs(data), 98)
    im = ax.imshow(data, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 2. Quick path (Lila)

Define one `MagneticSource`, attach it to the result, call `fit_sources()`:

---

In [ ]:
# Define a MagneticSource over the centre of the image
# center=(x, y) in pixels: x increases rightward, y increases downward
source = qdmpy.MagneticSource(
    name='grain_1',
    center=(32.0, 32.0),    # centre pixel (x=col, y=row)
    half_extent=(8.0, 8.0), # +-8 pixels in each direction -> 17x17 ROI
    pixel_spacing=4e-6,     # must match result.pixel_spacing
    model=qdmpy.MagneticModel(
        inclination=45.0,       # degrees below horizontal plane; range [-90, 90]
        declination=90.0,       # azimuthal angle; 90 = +X (East)
        magnetic_moment=1e-14,  # A*m^2; initial guess
    ),
)

print(f"Source name: {source.name}")
print(f"ROI: {source.roi_pixels}")
print(f"ROI size: {source.roi_pixels[0].stop - source.roi_pixels[0].start} x "
      f"{source.roi_pixels[1].stop - source.roi_pixels[1].start} pixels")
print(f"Centre: {source.center_um[0]:.0f} µm, {source.center_um[1]:.0f} µm")

In [ ]:
# Attach source to result (QDMResult is a Pydantic model — use model_copy)
result_with_source = result.model_copy(update={'field_sources': [source]})

# Fit — returns list[FitSourceResult], one per MagneticSource
fit_results = qdmpy.fit_sources(result_with_source, standoff_m=5e-6)
fsr = fit_results[0]

print(f"Converged: {fsr.raw.success}")
print(f"Cost (Huber): {fsr.raw.cost:.4e}")
print()
print("Fitted model:")
print(f"  magnetic_moment: {fsr.source.model.magnetic_moment:.3e} A*m^2")
print(f"  inclination:     {fsr.source.model.inclination:.1f} deg")
print(f"  declination:     {fsr.source.model.declination:.1f} deg")

## 3. Deep path (Professor)

### Comparing predicted vs measured Bz

`qdmpy.compute_field()` evaluates the analytical dipole formula over the ROI
grid using the **fitted** model, returning Bz in Tesla.

Compare with the measured Bz ROI to check residuals:

---

In [ ]:
# Extract measured Bz over the ROI (in Tesla)
bz_T = bz_uT * 1e-6                           # convert µT -> T
measured_roi = bz_T[source.roi_pixels]         # (roi_H, roi_W)

# Predicted Bz from the fitted dipole model
predicted = qdmpy.compute_field(fsr.source, standoff_m=5e-6)   # (roi_H, roi_W) in Tesla

residual = measured_roi - predicted

print(f"Measured ROI  — mean: {measured_roi.mean():.3e} T, std: {measured_roi.std():.3e} T")
print(f"Predicted ROI — mean: {predicted.mean():.3e} T, std: {predicted.std():.3e} T")
print(f"Residual      — mean: {residual.mean():.3e} T, std: {residual.std():.3e} T")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

titles = ['Measured Bz ROI (T)', 'Fitted dipole Bz (T)', 'Residual (T)']
arrays = [measured_roi, predicted, residual]

# Use same scale for measured and predicted; separate scale for residual
scales = [
    np.percentile(np.abs(measured_roi), 99),
    np.percentile(np.abs(measured_roi), 99),
    np.percentile(np.abs(residual), 99),
]

for ax, arr, title, vmax in zip(axes, arrays, titles, scales, strict=False):
    im = ax.imshow(arr, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle(f"grain_1  |  moment={fsr.source.model.magnetic_moment:.2e} A·m²  "
             f" inc={fsr.source.model.inclination:.0f}°  "
             f" dec={fsr.source.model.declination:.0f}°")
plt.tight_layout()
plt.show()

### Convergence diagnostics

The raw `scipy.OptimizeResult` exposes the full convergence information:

---

In [ ]:
raw = fsr.raw

print("Convergence summary:")
print(f"  success:     {raw.success}")
print(f"  cost:        {raw.cost:.4e}   (sum of Huber-weighted squared residuals)")
print(f"  nfev:        {raw.nfev}       (function evaluations)")
print(f"  message:     {raw.message}")
print()
print("Fitted parameters [x_offset_m, y_offset_m, z_m, mx, my, mz]:")
print(f"  {raw.x}")

## 4. Multi-source fitting

`fit_sources()` iterates over all `MagneticSource` objects in
`result.field_sources`. Define non-overlapping ROIs and pass them all at once:

---

In [ ]:
source_a = qdmpy.MagneticSource(
    name='grain_A',
    center=(16.0, 16.0),
    half_extent=(6.0, 6.0),
    pixel_spacing=4e-6,
    model=qdmpy.MagneticModel(
        inclination=30.0,
        declination=45.0,
        magnetic_moment=5e-15,
    ),
)

source_b = qdmpy.MagneticSource(
    name='grain_B',
    center=(48.0, 48.0),
    half_extent=(6.0, 6.0),
    pixel_spacing=4e-6,
    model=qdmpy.MagneticModel(
        inclination=60.0,
        declination=180.0,
        magnetic_moment=8e-15,
    ),
)

result_multi = result.model_copy(update={'field_sources': [source_a, source_b]})
fit_results_multi = qdmpy.fit_sources(result_multi, standoff_m=5e-6)

print(f"Fitted {len(fit_results_multi)} sources:")
for fsr_i in fit_results_multi:
    m = fsr_i.source.model
    print(
        f"  {fsr_i.source.name}: "
        f"moment={m.magnetic_moment:.3e} A*m^2  "
        f"inc={m.inclination:.1f} deg  "
        f"dec={m.declination:.1f} deg  "
        f"success={fsr_i.raw.success}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

vmax = np.percentile(np.abs(bz_uT), 98)
for ax, fsr_i in zip(axes, fit_results_multi, strict=False):
    s = fsr_i.source
    roi_row, roi_col = s.roi_pixels
    roi_data = bz_uT[roi_row, roi_col]
    im = ax.imshow(roi_data, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(
        f"{s.name}\nmoment={s.model.magnetic_moment:.2e} A·m²"
    )
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

## Key takeaways

- `MagneticSource` defines the ROI and initial dipole model priors; the fitter
  optimises (x_offset, y_offset, z, mx, my, mz) simultaneously
- Attach sources via `result.model_copy(update={'field_sources': [...]})`
- `fit_sources(result, standoff_m)` returns one `FitSourceResult` per
  `MagneticSource`; inspect `.raw.success` and `.raw.cost` for convergence
- `compute_field(fitted_source, standoff_m)` gives the predicted Bz (Tesla)
  for residual analysis
- Multi-source fitting: pass a list of non-overlapping sources; each is fitted
  independently

---

## What's next

- **Professor** — [Extending QDMpy](../extending.md) to implement custom
  `FieldReconstructor` or add new `FieldSource` subclasses
- **Lila** — [Fitting Quality](fitting.md) to understand chi2 and
  fit_states before interpreting source fitting results
- **API reference** — [`qdmpy.source_fitting`](../api/index.md) for full
  parameter documentation

---

> **Note on synthetic data:** The dipole field in `make_synthetic_qdm_result`
> is a simple analytical approximation, not a true point dipole. The fitter
> will converge to the best single-dipole approximation of the ROI, but the
> residuals will be non-zero. With real QDM data from a ferromagnetic grain,
> residuals should be much smaller for a well-isolated source.